In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

np.random.seed(42)
random.seed(42)

# ------------------------------
# 1. Загрузка и подготовка FedCycleData (основа циклов)
# ------------------------------
print("Загружаем FedCycleData...")
fed = pd.read_csv('FedCycleData071012 (2).csv', encoding='utf-8')

# Сохраняем нужные колонки, включая антропометрию
fed = fed[['ClientID', 'CycleNumber', 'LengthofCycle', 
           'EstimatedDayofOvulation', 'LengthofLutealPhase', 
           'LengthofMenses', 'Age', 'Weight', 'Height']].copy()

# Удаляем строки с пропусками в критических колонках (без них нельзя построить цикл)
fed.dropna(subset=['LengthofCycle', 'EstimatedDayofOvulation', 
                   'LengthofLutealPhase', 'LengthofMenses'], inplace=True)

# Приводим к числовому типу
fed['LengthofCycle'] = pd.to_numeric(fed['LengthofCycle'], errors='coerce')
fed['EstimatedDayofOvulation'] = pd.to_numeric(fed['EstimatedDayofOvulation'], errors='coerce')
fed['LengthofLutealPhase'] = pd.to_numeric(fed['LengthofLutealPhase'], errors='coerce')
fed['LengthofMenses'] = pd.to_numeric(fed['LengthofMenses'], errors='coerce')
fed['Age'] = pd.to_numeric(fed['Age'], errors='coerce')
fed['Weight'] = pd.to_numeric(fed['Weight'], errors='coerce')
fed['Height'] = pd.to_numeric(fed['Height'], errors='coerce')

# Ещё раз удаляем строки, если после преобразования появились NaN в критических колонках
fed.dropna(subset=['LengthofCycle', 'EstimatedDayofOvulation', 
                   'LengthofLutealPhase', 'LengthofMenses'], inplace=True)

# --- Заполнение пропусков в антропометрии ---
# Сортируем, чтобы forward fill работал по порядку циклов
fed.sort_values(['ClientID', 'CycleNumber'], inplace=True)

# Для каждого пользователя заполняем пропуски предыдущими значениями (вперёд и назад)
fed['Age'] = fed.groupby('ClientID')['Age'].transform(lambda x: x.ffill().bfill())
fed['Weight'] = fed.groupby('ClientID')['Weight'].transform(lambda x: x.ffill().bfill())
fed['Height'] = fed.groupby('ClientID')['Height'].transform(lambda x: x.ffill().bfill())

# Если после этого остались пропуски (например, пользователь без единой записи), заполняем средними
fed['Age'].fillna(fed['Age'].mean(), inplace=True)
fed['Weight'].fillna(fed['Weight'].mean(), inplace=True)
fed['Height'].fillna(fed['Height'].mean(), inplace=True)

# Преобразуем в int для нужных колонок
fed[['LengthofCycle', 'EstimatedDayofOvulation', 'LengthofLutealPhase', 'LengthofMenses']] = \
    fed[['LengthofCycle', 'EstimatedDayofOvulation', 'LengthofLutealPhase', 'LengthofMenses']].astype(int)

print(f"Загружено {len(fed)} циклов.")

records = []
for idx, row in fed.iterrows():
    user = row['ClientID']
    cycle = row['CycleNumber']
    cycle_len = row['LengthofCycle']
    ovul_day = row['EstimatedDayofOvulation']
    menses_len = row['LengthofMenses']
    
    for day in range(1, cycle_len + 1):
        if day <= menses_len:
            phase = 'menstrual'
        elif day < ovul_day:
            phase = 'follicular'
        elif day == ovul_day:
            phase = 'ovulatory'
        else:
            phase = 'luteal'
        records.append({
            'user_id': user,
            'cycle_id': cycle,
            'day_in_cycle': day,
            'phase': phase,
            'cycle_len': cycle_len,
            'ovulation_day': ovul_day
        })

df_cycles = pd.DataFrame(records)
print(f"Создано {len(df_cycles)} записей (дней циклов).")

# ------------------------------
# 2. Загрузка данных о шагах: индивидуальные средние по фазам
# ------------------------------
print("\nЗагружаем user_delta_fol_minus_men.csv...")
user_delta = pd.read_csv('user_delta_fol_minus_men.csv', encoding='utf-8')

rename_map = {}
for col in user_delta.columns:
    col_lower = col.strip().lower()
    if 'id' in col_lower:
        rename_map[col] = 'user_id_original'
    elif 'follicular' in col_lower:
        rename_map[col] = 'steps_follicular'
    elif 'luteal' in col_lower:
        rename_map[col] = 'steps_luteal'
    elif 'menstrual' in col_lower:
        rename_map[col] = 'steps_menstrual'
    elif 'ovulatory' in col_lower:
        rename_map[col] = 'steps_ovulatory'

if rename_map:
    user_delta.rename(columns=rename_map, inplace=True)

required_step_cols = ['steps_follicular', 'steps_luteal', 'steps_menstrual', 'steps_ovulatory']


step_data = user_delta[required_step_cols].dropna()
phase_means = step_data.mean().to_dict()
phase_cov = step_data.cov().to_numpy()
phase_cov += np.eye(4) * 1e-6

print("Средние шагов по фазам из user_delta:")
print(phase_means)

unique_users = df_cycles['user_id'].unique()
n_users = len(unique_users)
generated_means = np.random.multivariate_normal(
    mean=list(phase_means.values()),
    cov=phase_cov,
    size=n_users
)
generated_means = np.maximum(generated_means, 500)  # минимум 500 шагов

user_phase_steps = {}
for i, user in enumerate(unique_users):
    user_phase_steps[user] = {
        'follicular': generated_means[i, 0],
        'luteal': generated_means[i, 1],
        'menstrual': generated_means[i, 2],
        'ovulatory': generated_means[i, 3]
    }

# Корректировка аномально высоких менструальных шагов
print("Корректировка соотношений фаз...")
correction_count = 0
for user in user_phase_steps:
    fol = user_phase_steps[user]['follicular']
    men = user_phase_steps[user]['menstrual']
    if men > 1.2 * fol:
        old_men = men
        user_phase_steps[user]['menstrual'] = fol * 1.1
        correction_count += 1
print(f"Скорректировано {correction_count} пользователей.")
    
print("Статистика сгенерированных средних шагов по фазам:")
df_check = pd.DataFrame(user_phase_steps).T
print(df_check.describe())


# ------------------------------
# 3. Загрузка дневного профиля шагов (cycleday_trend)
# ------------------------------
print("\nЗагружаем cycleday_trend.csv...")
trend = pd.read_csv('cycleday_trend.csv')
trend.set_index('CycleDay', inplace=True)
max_day = df_cycles['day_in_cycle'].max()
all_days = np.arange(1, max_day+1)
trend_interp = trend.reindex(all_days).interpolate(method='linear', limit_direction='both')
daily_profile = trend_interp['avg_steps'].values
daily_profile_norm = daily_profile / daily_profile.mean()

# ------------------------------
# 4. Генерация ежедневных шагов
# ------------------------------
df_cycles['steps'] = 0

for (user, cycle), group in df_cycles.groupby(['user_id', 'cycle_id']):
    user_targets = user_phase_steps[user]
    for phase in ['menstrual', 'follicular', 'ovulatory', 'luteal']:
        phase_days = group[group['phase'] == phase]['day_in_cycle'].values
        if len(phase_days) == 0:
            continue
        mean_profile_weight = daily_profile_norm[phase_days - 1].mean()
        target = user_targets[phase]
        if mean_profile_weight <= 0:
            scale = 1.0
        else:
            scale = target / mean_profile_weight
        
        mask = (df_cycles['user_id'] == user) & (df_cycles['cycle_id'] == cycle) & (df_cycles['phase'] == phase)
        days_indices = df_cycles.loc[mask, 'day_in_cycle'].values - 1
        base_weights = daily_profile_norm[days_indices]
        noise = np.random.normal(0, 0.2 * target, size=len(days_indices))
        steps = base_weights * scale + noise
        steps = np.maximum(0, steps).astype(int)
        df_cycles.loc[mask, 'steps'] = steps

print("Шаги сгенерированы.")

# ------------------------------
# 5. Генерация калорий с учётом BMR (на основе реальных данных, с конвертацией единиц)
# ------------------------------

# 5.1 Извлекаем для каждого пользователя его возраст, вес (фунты), рост (дюймы) из FedCycleData
user_age = {}
user_weight_lb = {}   # вес в фунтах
user_height_in = {}   # рост в дюймах

for user in unique_users:
    user_rows = fed[fed['ClientID'] == user]
    if len(user_rows) > 0:
        # Берём первое значение (предполагаем, что они одинаковы для всех циклов)
        age_val = user_rows['Age'].iloc[0]
        weight_val = user_rows['Weight'].iloc[0]
        height_val = user_rows['Height'].iloc[0]
        
        # Проверяем, что значения не пропущены и могут быть преобразованы в число
        user_age[user] = float(age_val) if pd.notna(age_val) else np.nan
        user_weight_lb[user] = float(weight_val) if pd.notna(weight_val) else np.nan
        user_height_in[user] = float(height_val) if pd.notna(height_val) else np.nan
    else:
        user_age[user] = np.nan
        user_weight_lb[user] = np.nan
        user_height_in[user] = np.nan

# Заполняем пропуски средними значениями (в исходных единицах)
mean_age = fed['Age'].mean()
mean_weight_lb = fed['Weight'].mean()
mean_height_in = fed['Height'].mean()

for user in unique_users:
    if pd.isna(user_age[user]):
        user_age[user] = mean_age
    if pd.isna(user_weight_lb[user]):
        user_weight_lb[user] = mean_weight_lb
    if pd.isna(user_height_in[user]):
        user_height_in[user] = mean_height_in

# 5.2 Функция конвертации в метрические и расчёта BMR
def calculate_bmr_from_imperial(weight_lb, height_in, age_years):
    # Конвертация
    weight_kg = weight_lb * 0.45359237
    height_cm = height_in * 2.54
    # Формула Миффлина-Сан Жеора для женщин
    bmr = 10 * weight_kg + 6.25 * height_cm - 5 * age_years - 161
    return bmr

# 5.3 Загружаем целевые калории из phase_summary 
print("\nЗагружаем phase_summary.csv для калорий...")
try:
    phase_summary_cal = pd.read_csv('phase_summary.csv')
    phase_summary_cal.set_index('CyclePhase', inplace=True)
    if 'avg_cal' in phase_summary_cal.columns:
        cal_target = phase_summary_cal['avg_cal'].to_dict()
    else:
        cal_target = {'menstrual': 1800, 'follicular': 2000, 'ovulatory': 2100, 'luteal': 1900}
except:
    cal_target = {'menstrual': 1800, 'follicular': 2000, 'ovulatory': 2100, 'luteal': 1900}

# 5.4 Для каждого пользователя и фазы вычисляем индивидуальный коэффициент активности
user_activity_factor = {}
for user in unique_users:
    user_activity_factor[user] = {}
    # Рассчитываем BMR для этого пользователя
    bmr = calculate_bmr_from_imperial(user_weight_lb[user], user_height_in[user], user_age[user])
    # На случай, если BMR получился нереалистичным (например, из-за пропусков), ставим разумный минимум
    if bmr < 1000:
        bmr = 1400  # средний BMR для женщин
    for phase in ['menstrual', 'follicular', 'ovulatory', 'luteal']:
        target_tdee = cal_target.get(phase, 2000)
        target_steps = user_phase_steps[user][phase]
        
        # Энергия, которая должна приходиться на активность (и TEF)
        activity_energy = target_tdee - bmr
        if activity_energy < 50:   # слишком мало – ставим минимум
            activity_energy = 50
        
        if target_steps > 0:
            a = activity_energy / target_steps
        else:
            a = 0.03  # значение по умолчанию
        
        user_activity_factor[user][phase] = a

# 5.5 Генерация калорий для каждой строки
cal_noise_sd = 50

def generate_calories_with_bmr(row):
    steps = row['steps']
    if pd.isna(steps):
        return np.nan
    user = row['user_id']
    phase = row['phase']
    
    # Получаем BMR пользователя
    bmr = calculate_bmr_from_imperial(user_weight_lb[user], user_height_in[user], user_age[user])
    if bmr < 1000:
        bmr = 1400
    
    a = user_activity_factor[user][phase]
    
    cal = bmr + a * steps + np.random.normal(0, cal_noise_sd)
    return max(1000, int(cal))

# Применяем функцию
df_cycles['calories'] = df_cycles.apply(generate_calories_with_bmr, axis=1)

# ------------------------------
# 6. Генерация ЧСС и ВСР на основе реальных данных Whoop_Fitness_Dataset
# ------------------------------

import pandas as pd
import numpy as np
import statsmodels.api as sm

# Загрузка и фильтрация данных Whoop_Fitness_Dataset
df_whoop = pd.read_csv('whoop_fitness_dataset_100k.csv')
df_female = df_whoop[df_whoop['gender'] == 'Female'].copy()

# Группировка по пользователям
female_stats = df_female.groupby('user_id').agg(
    rhr_mean=('resting_heart_rate', 'mean'),
    rhr_std=('resting_heart_rate', 'std'),
    hrv_mean=('hrv', 'mean'),
    hrv_std=('hrv', 'std'),
    age=('age', 'first'),
    weight_kg=('weight_kg', 'first'),
    height_cm=('height_cm', 'first')
).reset_index()

# Удаляем пользователей с пропущенными значениями
female_stats.dropna(subset=['rhr_mean', 'hrv_mean', 'age'], inplace=True)

# --- 1. Регрессия RHR от возраста ---
X = sm.add_constant(female_stats['age'])
y = female_stats['rhr_mean']
model = sm.OLS(y, X).fit()
age_rhr_intercept = model.params['const']
age_rhr_slope = model.params['age']
print(f"RHR регрессия: intercept = {age_rhr_intercept:.2f}, slope = {age_rhr_slope:.3f}")

# --- 2. Логнормальные параметры для HRV ---
log_hrv = np.log(female_stats['hrv_mean'])
hrv_logmean = log_hrv.mean()
hrv_logstd = log_hrv.std()
print(f"HRV логнормальные: mean = {hrv_logmean:.3f}, std = {hrv_logstd:.3f}")

# --- 3. Оценка внутрииндивидуальной вариабельности (шум) ---
# Берём средние значения стандартных отклонений, отбрасывая NaN
rhr_std_mean = female_stats['rhr_std'].dropna().mean()
hrv_std_mean = female_stats['hrv_std'].dropna().mean()
print(f"Среднее внутрииндивидуальное СКО: RHR = {rhr_std_mean:.2f}, HRV = {hrv_std_mean:.2f}")

# --- 4. Межиндивидуальный разброс средних ---
rhr_between_std = female_stats['rhr_mean'].std()
# Для HRV межиндивидуальный разброс уже учтён в логнормальном распределении

# Генерация базовых значений для каждого пользователя из FedCycleData
user_base_hr = {}
user_base_rmssd = {}

for user in unique_users:
    age = user_age.get(user, 30)
    
    # Предсказанный RHR с учётом возраста
    pred_rhr = age_rhr_intercept + age_rhr_slope * age
    # Добавляем межиндивидуальное отклонение
    base_hr = np.random.normal(pred_rhr, rhr_between_std)
    base_hr = np.clip(base_hr, 50, 90)
    user_base_hr[user] = base_hr
    
    # Базовый RMSSD из логнормального распределения
    log_rmssd = np.random.normal(hrv_logmean, hrv_logstd)
    base_rmssd = np.exp(log_rmssd)
    base_rmssd = np.clip(base_rmssd, 15, 150)
    user_base_rmssd[user] = base_rmssd

# Коэффициенты влияния фаз (на основе используемого метаанализа)
phase_hr_factor = {
    'menstrual': 1.0,
    'follicular': 0.96,
    'ovulatory': 1.02,
    'luteal': 1.05
}
phase_rmssd_factor = {
    'menstrual': 1.0,
    'follicular': 1.07,
    'ovulatory': 1.03,
    'luteal': 0.94
}

# Шумы (внутрииндивидуальная вариабельность)
hr_noise_sd = rhr_std_mean
rmssd_noise_sd = hrv_std_mean

def generate_hr(row):
    user = row['user_id']
    phase = row['phase']
    base = user_base_hr[user]
    factor = phase_hr_factor.get(phase, 1.0)
    hr = base * factor + np.random.normal(0, hr_noise_sd)
    return int(round(np.clip(hr, 45, 100)))

def generate_rmssd(row):
    user = row['user_id']
    phase = row['phase']
    base = user_base_rmssd[user]
    factor = phase_rmssd_factor.get(phase, 1.0)
    rmssd = base * factor + np.random.normal(0, rmssd_noise_sd)
    return int(round(np.clip(rmssd, 10, 180)))

df_cycles['heart_rate'] = df_cycles.apply(generate_hr, axis=1)
df_cycles['rmssd'] = df_cycles.apply(generate_rmssd, axis=1)

# Проверка результатов
print("\nСредние значения по фазам:")
print(df_cycles.groupby('phase')[['heart_rate', 'rmssd']].mean())
print("\nСтандартные отклонения по фазам:")
print(df_cycles.groupby('phase')[['heart_rate', 'rmssd']].std())

# ------------------------------
# 7. Генерация продолжительности сна
# ------------------------------
print("\nЗагружаем Period_Log.csv...")
try:
    period_log = pd.read_csv('Period_Log.csv')
    if 'sleep_hours_cycle' in period_log.columns and 'cycle_phase' in period_log.columns:
        sleep_stats = period_log.groupby('cycle_phase')['sleep_hours_cycle'].agg(['mean', 'std']).to_dict('index')
        sleep_stats_lower = {k.lower(): v for k, v in sleep_stats.items()}
    else:
        sleep_stats_lower = {}
except:
    sleep_stats_lower = {}

default_sleep_mean = 7.5
default_sleep_std = 1.0

def generate_sleep(row):
    phase = row['phase']
    stats = sleep_stats_lower.get(phase, {'mean': default_sleep_mean, 'std': default_sleep_std})
    mean = stats['mean']
    std = stats['std']
    sleep = np.random.normal(mean, std)
    return round(max(3, min(12, sleep)), 1)

df_cycles['sleep_hours'] = df_cycles.apply(generate_sleep, axis=1)

# ------------------------------
# 8. Генерация субъективных показателей с зависимостями
# ------------------------------

# Словарь показателей и их диапазонов 
subj_cols = {
    'pain_level': (1, 10),
    'mood_score': (1, 10),
    'stress_score_cycle': (1, 10),
    'energy_level': (1, 10),
    'concentration_score': (1, 10)
}

# Загружаем статистики из Period_Log.csv, если они есть 
subj_stats = {}
for col in subj_cols.keys():
    if col in period_log.columns and 'cycle_phase' in period_log.columns:
        stats = period_log.groupby('cycle_phase')[col].agg(['mean', 'std']).to_dict('index')
        subj_stats[col] = {k.lower(): v for k, v in stats.items()}
    else:
        print(f"Колонка {col} не найдена в Period_Log. Используем значения по умолчанию.")
        subj_stats[col] = {}  # пустой словарь – позже будут использованы дефолтные 5 и 2

# Вспомогательная функция для получения среднего и стандартного отклонения по фазе
def get_phase_stats(col, phase):
    if col in subj_stats and phase in subj_stats[col]:
        return subj_stats[col][phase]['mean'], subj_stats[col][phase]['std']
    else:
        return 5.0, 2.0  # значения по умолчанию

# --- 8.1 Уровень стресса (зависит от ВСР, ЧСС и сна) ---
def generate_stress(row):
    phase = row['phase']
    base_mean, base_std = get_phase_stats('stress_score_cycle', phase)
    
    user = row['user_id']
    # Нормированные отклонения от индивидуальных базовых уровней
    rmssd_dev = (row['rmssd'] - user_base_rmssd[user]) / user_base_rmssd[user]
    hr_dev = (row['heart_rate'] - user_base_hr[user]) / user_base_hr[user]
    # Отклонение продолжительности сна от среднего (7.5 ч)
    sleep_dev = row['sleep_hours'] - 7.5
    
    # Влияние: низкая ВСР → стресс выше, высокая ЧСС → стресс выше, недосып → стресс выше
    adjustment = -2.0 * rmssd_dev + 1.5 * hr_dev - 0.3 * sleep_dev
    adjustment = np.clip(adjustment, -2.5, 2.5)  # ограничиваем, чтобы не выйти за пределы шкалы
    
    value = np.random.normal(base_mean + adjustment, base_std)
    return int(round(np.clip(value, 1, 10)))

df_cycles['stress_score_cycle'] = df_cycles.apply(generate_stress, axis=1)

# --- 8.2 Уровень энергии (зависит от сна и стресса) ---
def generate_energy(row):
    phase = row['phase']
    base_mean, base_std = get_phase_stats('energy_level', phase)
    
    sleep_effect = (row['sleep_hours'] - 7.5) * 0.3          # больше сна → больше энергии
    stress_effect = -(row['stress_score_cycle'] - 5.5) * 0.2  # выше стресс → меньше энергии
    
    adjustment = sleep_effect + stress_effect
    adjustment = np.clip(adjustment, -2.0, 2.0)
    
    value = np.random.normal(base_mean + adjustment, base_std)
    return int(round(np.clip(value, 1, 10)))

df_cycles['energy_level'] = df_cycles.apply(generate_energy, axis=1)

# --- 8.3 Настроение (зависит от стресса и сна) ---
def generate_mood(row):
    phase = row['phase']
    base_mean, base_std = get_phase_stats('mood_score', phase)
    
    stress_effect = -(row['stress_score_cycle'] - 5.5) * 0.25
    sleep_effect = (row['sleep_hours'] - 7.5) * 0.2
    
    adjustment = stress_effect + sleep_effect
    adjustment = np.clip(adjustment, -2.0, 2.0)
    
    value = np.random.normal(base_mean + adjustment, base_std)
    return int(round(np.clip(value, 1, 10)))

df_cycles['mood_score'] = df_cycles.apply(generate_mood, axis=1)

# --- 8.4 Концентрация (зависит от стресса и энергии) ---
def generate_concentration(row):
    phase = row['phase']
    base_mean, base_std = get_phase_stats('concentration_score', phase)
    
    stress_effect = -(row['stress_score_cycle'] - 5.5) * 0.2
    energy_effect = (row['energy_level'] - 5.5) * 0.3
    
    adjustment = stress_effect + energy_effect
    adjustment = np.clip(adjustment, -2.0, 2.0)
    
    value = np.random.normal(base_mean + adjustment, base_std)
    return int(round(np.clip(value, 1, 10)))

df_cycles['concentration_score'] = df_cycles.apply(generate_concentration, axis=1)

# --- 8.5 Уровень боли (зависит от фазы и стресса) ---
def generate_pain(row):
    phase = row['phase']
    base_mean, base_std = get_phase_stats('pain_level', phase)
    
    # Стресс может усиливать восприятие боли
    stress_effect = (row['stress_score_cycle'] - 5.5) * 0.15
    adjustment = np.clip(stress_effect, -1.0, 1.0)
    
    value = np.random.normal(base_mean + adjustment, base_std)
    return int(round(np.clip(value, 1, 10)))

df_cycles['pain_level'] = df_cycles.apply(generate_pain, axis=1)



# ------------------------------
# 10. Добавление пропусков и артефактов
# ------------------------------
missing_prob = 0.05
mask_missing = np.random.random(len(df_cycles)) < missing_prob
cols_to_miss = ['steps', 'calories', 'heart_rate', 'rmssd', 'sleep_hours'] + list(subj_cols.keys())
df_cycles.loc[mask_missing, cols_to_miss] = np.nan

outlier_prob = 0.01
mask_outlier = np.random.random(len(df_cycles)) < outlier_prob
df_cycles.loc[mask_outlier, 'heart_rate'] = df_cycles.loc[mask_outlier, 'heart_rate'] * np.random.uniform(1.2, 1.5)
df_cycles.loc[mask_outlier, 'rmssd'] = df_cycles.loc[mask_outlier, 'rmssd'] * np.random.uniform(0.5, 0.8)

# ------------------------------
# 11. Сохранение
# ------------------------------
output_file = 'synthetic_menstrual_physiology_with_subjective.csv'
df_cycles.to_csv(output_file, index=False)
print(f"\nСинтетический датасет сохранён в файл: {output_file}")
print(f"Всего записей: {len(df_cycles)}")
print("Первые 5 строк:")
print(df_cycles.head())

Загружаем FedCycleData...
Загружено 1512 циклов.
Создано 44220 записей (дней циклов).

Загружаем user_delta_fol_minus_men.csv...
Средние шагов по фазам из user_delta:
{'steps_follicular': 7370.147324628776, 'steps_luteal': 7975.849707469866, 'steps_menstrual': 7370.672887864823, 'steps_ovulatory': 7628.505376344086}
Корректировка соотношений фаз...
Скорректировано 37 пользователей.
Статистика сгенерированных средних шагов по фазам:
         follicular        luteal     menstrual     ovulatory
count    157.000000    157.000000    157.000000    157.000000
mean    7413.749324   7991.408743   7063.357317   7799.584431
std     3592.487187   3401.017820   3648.171251   3903.372709
min      500.000000    500.000000    500.000000    500.000000
25%     4784.367039   5596.939199   4617.760536   4596.828332
50%     7542.511125   7862.090465   6982.552831   8531.226660
75%     9965.913919  10446.773477   9612.033939  10634.241391
max    15723.054094  15970.585919  16250.648655  19353.046565

Загру